# Test ingestion + retrieval endpoints

Start the service first:

```powershell
uv run python main.py
```

In [ ]:
import json
import requests

BASE_URL = "http://localhost:9000"

# Must match RETRIEVAL_API_KEY in the server's .env. Sent on every protected call.
API_KEY = "change-me"
HEADERS = {"X-API-Key": API_KEY}

def pretty(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False))

## Health

In [ ]:
r = requests.get(f"{BASE_URL}/health", timeout=10)
r.raise_for_status()
pretty(r.json())

## Ingest a batch of YouTube videos (up to 2 in flight server-side)

In [ ]:
VIDEO_URLS = [
    "https://www.youtube.com/watch?v=r6EXZcTJyaA",
    "https://www.youtube.com/watch?v=gaa_5m3cmSc",
]

r = requests.post(f"{BASE_URL}/ingest", json={"urls": VIDEO_URLS}, headers=HEADERS, timeout=1200)
r.raise_for_status()
ingest_result = r.json()
pretty(ingest_result)

VIDEO_IDS = [item["video_id"] for item in ingest_result["results"] if item["video_id"]]
print("VIDEO_IDS =", VIDEO_IDS)

## Retrieve: metadata mode

For queries like "which video performed better?" — returns per-video metadata only, no chunks.

In [ ]:
r = requests.post(
    f"{BASE_URL}/retrieve",
    json={"mode": "metadata", "video_ids": VIDEO_IDS},
    headers=HEADERS,
    timeout=30,
)
r.raise_for_status()
pretty(r.json())

## Retrieve: chunks mode (hybrid + RRF + cross-encoder rerank)

Pipeline: Dense (Qdrant) + BM25 → RRF → top-15 candidates → cross-encoder rerank → top-3.
Pass `video_ids` to scope retrieval to a subset of indexed videos.

In [ ]:
QUERY = "timestamp where he says he will kidnap his granddaughter"

r = requests.post(
    f"{BASE_URL}/retrieve",
    json={
        "mode": "chunks",
        "query": QUERY,
        "video_ids": VIDEO_IDS or None,
        "candidate_k": 15,
        "top_k": 3,
    },
    headers=HEADERS,
    timeout=60,
)
r.raise_for_status()
retrieval = r.json()
pretty(retrieval)

In [ ]:
for i, hit in enumerate(retrieval["results"], start=1):
    md = hit["metadata"]
    snippet = hit["text"][:220].replace("\n", " ")
    print(
        f"#{i} rerank={hit['rerank_score']:.4f} rrf={hit['rrf_score']:.4f}  "
        f"[{md.get('start_time')}-{md.get('end_time')}]  {md.get('title')}"
    )
    print(f"     {snippet}...\n")

## Sanity sweep across a few queries

In [ ]:
queries = [
    "what is the main topic of the video",
    "any mention of performance or optimization",
    "opening hook or introduction",
]

for q in queries:
    r = requests.post(
        f"{BASE_URL}/retrieve",
        json={"mode": "chunks", "query": q, "candidate_k": 15, "top_k": 3},
        headers=HEADERS,
        timeout=60,
    )
    r.raise_for_status()
    res = r.json()["results"]
    print(f"\n=== {q} ===")
    for i, hit in enumerate(res, start=1):
        md = hit["metadata"]
        snippet = hit["text"][:160].replace("\n", " ")
        print(f"  #{i} rerank={hit['rerank_score']:.4f} rrf={hit['rrf_score']:.4f} [{md.get('start_time')}-{md.get('end_time')}] {snippet}...")